In [1]:
import os
import glob
import wfdb

MIMIC_ROOT = "/home/asatsan2/Projects/ECG/mimic_ecg/p1000only"

hea_files = sorted(glob.glob(os.path.join(MIMIC_ROOT, "**", "*.hea"), recursive=True))
print("num hea:", len(hea_files))

for hp in hea_files[:5]:
    rp = hp[:-4]
    print("\n" + "="*80)
    print("FILE:", hp)
    try:
        rec = wfdb.rdrecord(rp)
        print("comments:", rec.comments)
        print("sig_name:", rec.sig_name)
        print("fs:", rec.fs)
    except Exception as e:
        print("ERROR:", e)
        with open(hp, "r") as f:
            txt = f.read()
        print("RAW HEADER:\n", txt[:1500])
        

num hea: 0


In [2]:
import os
import glob

MIMIC_ROOT = "/home/asatsan2/Projects/ECG/mimic_ecg/p1000"

hea_files = sorted(glob.glob(os.path.join(MIMIC_ROOT, "**", "*.hea"), recursive=True))
dat_files = sorted(glob.glob(os.path.join(MIMIC_ROOT, "**", "*.dat"), recursive=True))

print("hea count:", len(hea_files))
print("dat count:", len(dat_files))
print("first hea:", hea_files[:3])
print("first dat:", dat_files[:3])


hea count: 795
dat count: 795
first hea: ['/home/asatsan2/Projects/ECG/mimic_ecg/p1000/p10000032/s40689238/40689238.hea', '/home/asatsan2/Projects/ECG/mimic_ecg/p1000/p10000032/s44458630/44458630.hea', '/home/asatsan2/Projects/ECG/mimic_ecg/p1000/p10000032/s49036311/49036311.hea']
first dat: ['/home/asatsan2/Projects/ECG/mimic_ecg/p1000/p10000032/s40689238/40689238.dat', '/home/asatsan2/Projects/ECG/mimic_ecg/p1000/p10000032/s44458630/44458630.dat', '/home/asatsan2/Projects/ECG/mimic_ecg/p1000/p10000032/s49036311/49036311.dat']


In [4]:
import os
import re
import glob
import random
import numpy as np
import pandas as pd

MIMIC_ROOT = "/home/asatsan2/Projects/ECG/mimic_ecg"
MAX_SAMPLES = 200
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

def build_mimic_metadata_from_csv(mimic_root, max_samples=200, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    record_csv = os.path.join(mimic_root, "record_list.csv")
    mm_csv = os.path.join(mimic_root, "machine_measurements.csv")

    record_df = pd.read_csv(record_csv)
    mm_df = pd.read_csv(mm_csv)

    print("record_list shape:", record_df.shape)
    print("machine_measurements shape:", mm_df.shape)

    # Find all locally downloaded .hea files
    hea_files = sorted(glob.glob(os.path.join(mimic_root, "p*", "**", "*.hea"), recursive=True))
    local_record_paths = [hp[:-4] for hp in hea_files]  # remove .hea
    local_rel_paths = [os.path.relpath(p, mimic_root) for p in local_record_paths]

    print("Local downloaded .hea files:", len(hea_files))

    # Find waveform path column in record_list.csv
    candidate_cols = ["path", "record_path", "file_name", "wfdb_path"]
    path_col = None
    for c in candidate_cols:
        if c in record_df.columns:
            path_col = c
            break

    if path_col is None:
        raise ValueError(f"Could not find path column in record_list.csv. Columns: {record_df.columns.tolist()}")

    record_df[path_col] = record_df[path_col].astype(str).str.replace(r"\.hea$|\.dat$", "", regex=True)

    local_set = set(local_rel_paths)
    record_df = record_df[record_df[path_col].isin(local_set)].copy()

    print("Matched locally available records:", len(record_df))

    # Join with machine_measurements.csv
    needed_join_cols = ["subject_id", "study_id"]
    for c in needed_join_cols:
        if c not in record_df.columns or c not in mm_df.columns:
            raise ValueError(f"Missing join column {c}")

    meta = record_df.merge(mm_df, on=["subject_id", "study_id"], how="left")
    print("Merged shape:", meta.shape)

    # Build text from machine-generated report fields
    report_cols = [c for c in meta.columns if re.fullmatch(r"report_\d+", str(c))]
    report_cols = sorted(report_cols, key=lambda x: int(x.split("_")[1]))

    if len(report_cols) == 0:
        raise ValueError("No report_# columns found in machine_measurements.csv")

    def build_machine_text(row):
        parts = []
        for c in report_cols:
            val = row.get(c, None)
            if pd.notna(val) and str(val).strip():
                parts.append(str(val).strip())
        return " ".join(parts).strip()

    meta["report"] = meta.apply(build_machine_text, axis=1)
    meta["report"] = meta["report"].fillna("").astype(str)
    meta["report"] = meta["report"].str.strip()
    meta = meta[meta["report"] != ""].copy()

    print("Rows with usable machine text:", len(meta))

    # Label mapping from machine text
    def map_label(text):
        t = str(text).lower()

        if any(x in t for x in ["normal sinus rhythm", "normal ecg", "sinus rhythm", "within normal limits"]):
            return "NORM"
        if any(x in t for x in ["myocardial infarction", "infarct", "stemi", "nstemi", "anterior infarct", "inferior infarct"]):
            return "MI"
        if any(x in t for x in ["atrial fibrillation", "afib", "a-fib", "atrial fib"]):
            return "AFIB"
        if any(x in t for x in ["sinus tachycardia", "tachycardia", "supraventricular tachycardia", "svt"]):
            return "TACHY"
        if any(x in t for x in ["sinus bradycardia", "bradycardia"]):
            return "BRADY"
        return "OTHER"

    meta["chosen_label"] = meta["report"].apply(map_label)

    print("\nRaw label counts:")
    print(meta["chosen_label"].value_counts())

    # Keep labels with enough support
    vc = meta["chosen_label"].value_counts()
    keep = vc[vc >= 5].index.tolist()
    meta = meta[meta["chosen_label"].isin(keep)].copy()

    # Attach full path for wfdb
    meta["filename_lr"] = meta[path_col].apply(lambda x: os.path.join(mimic_root, x))

    # Balanced subset
    if len(meta) > max_samples:
        n_classes = meta["chosen_label"].nunique()
        per_class = max(1, max_samples // n_classes)
        chunks = []
        for lab in sorted(meta["chosen_label"].unique()):
            part = meta[meta["chosen_label"] == lab]
            chunks.append(part.sample(min(len(part), per_class), random_state=seed))
        meta = pd.concat(chunks).sample(frac=1, random_state=seed).reset_index(drop=True)

        if len(meta) > max_samples:
            meta = meta.sample(max_samples, random_state=seed).reset_index(drop=True)

    # Study-level split
    idx = np.arange(len(meta))
    np.random.shuffle(idx)
    n = len(idx)

    train_idx = set(idx[:int(0.7 * n)])
    val_idx = set(idx[int(0.7 * n):int(0.85 * n)])
    test_idx = set(idx[int(0.85 * n):])

    meta["split"] = [
        "train" if i in train_idx else ("val" if i in val_idx else "test")
        for i in range(len(meta))
    ]

    label_names = sorted(meta["chosen_label"].unique())
    label_map = {lab: i for i, lab in enumerate(label_names)}

    out_cols = ["subject_id", "study_id", "filename_lr", "report", "chosen_label", "split"]
    meta = meta[out_cols].reset_index(drop=True)

    print("\nFinal size:", len(meta))
    print("\nFinal label counts:")
    print(meta["chosen_label"].value_counts())
    print("\nSplit counts:")
    print(meta["split"].value_counts())
    print("\nLabel map:", label_map)

    return meta, label_map

mimic_meta, mimic_label_map = build_mimic_metadata_from_csv(
    MIMIC_ROOT,
    max_samples=200,
    seed=42
)

print(mimic_meta.head())


/tmp/ipykernel_188096/672672928.py:23: DtypeWarning: Columns (16,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  mm_df = pd.read_csv(mm_csv)


record_list shape: (800035, 5)
machine_measurements shape: (800035, 33)
Local downloaded .hea files: 796
Matched locally available records: 0
Merged shape: (0, 36)
Rows with usable machine text: 0

Raw label counts:
Series([], Name: count, dtype: int64)

Final size: 0

Final label counts:
Series([], Name: count, dtype: int64)

Split counts:
Series([], Name: count, dtype: int64)

Label map: {}
Empty DataFrame
Columns: [subject_id, study_id, filename_lr, report, chosen_label, split]
Index: []


In [5]:
import os
import glob
import re
import pandas as pd

CANDIDATE_ROOTS = [
    "/home/asatsan2/Projects/ECG/mimic_ecg",
    "/home/asatsan2/Projects/ECG/dataset/physionet.org/files/mimic-iv-ecg/1.0",
    "/home/asatsan2/Projects/ECG/dataset/physionet.org/files/mimic-iv-ecg/1.0/files",
]

def normalize_record_path(x):
    x = str(x).strip()
    x = re.sub(r"^\./", "", x)
    x = re.sub(r"^/", "", x)
    x = re.sub(r"^files/", "", x)
    x = re.sub(r"\.hea$|\.dat$", "", x)
    return x

record_csv = None
for root in CANDIDATE_ROOTS:
    candidate = os.path.join(root, "record_list.csv")
    if os.path.exists(candidate):
        record_csv = candidate
        csv_root = root
        break

if record_csv is None:
    raise ValueError("record_list.csv not found in candidate roots")

print("Using CSV root:", csv_root)
record_df = pd.read_csv(record_csv)

path_col = None
for c in ["path", "record_path", "file_name", "wfdb_path"]:
    if c in record_df.columns:
        path_col = c
        break

if path_col is None:
    raise ValueError(f"No path column found. Columns: {record_df.columns.tolist()}")

record_df["norm_path"] = record_df[path_col].astype(str).apply(normalize_record_path)

best_root = None
best_matches = -1

for root in CANDIDATE_ROOTS:
    hea_files = sorted(glob.glob(os.path.join(root, "p*", "**", "*.hea"), recursive=True))
    local_paths = [normalize_record_path(os.path.relpath(h[:-4], root)) for h in hea_files]
    local_set = set(local_paths)
    matches = record_df["norm_path"].isin(local_set).sum()
    print(f"Root: {root}")
    print(f"  local .hea count: {len(hea_files)}")
    print(f"  matched paths: {matches}")
    if matches > best_matches:
        best_matches = matches
        best_root = root

print("\nBest MIMIC_ROOT =", best_root)

Using CSV root: /home/asatsan2/Projects/ECG/mimic_ecg
Root: /home/asatsan2/Projects/ECG/mimic_ecg
  local .hea count: 796
  matched paths: 795
Root: /home/asatsan2/Projects/ECG/dataset/physionet.org/files/mimic-iv-ecg/1.0
  local .hea count: 0
  matched paths: 0
Root: /home/asatsan2/Projects/ECG/dataset/physionet.org/files/mimic-iv-ecg/1.0/files
  local .hea count: 0
  matched paths: 0

Best MIMIC_ROOT = /home/asatsan2/Projects/ECG/mimic_ecg


In [6]:
import os
import re
import glob
import random
import numpy as np
import pandas as pd

MIMIC_ROOT = "/home/asatsan2/Projects/ECG/mimic_ecg"
MAX_SAMPLES = 200
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

def normalize_record_path(x):
    x = str(x).strip()
    x = re.sub(r"^\./", "", x)
    x = re.sub(r"^/", "", x)
    x = re.sub(r"^files/", "", x)
    x = re.sub(r"\.hea$|\.dat$", "", x)
    return x

def build_mimic_metadata_from_csv(mimic_root, max_samples=200, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    record_csv = os.path.join(mimic_root, "record_list.csv")
    mm_csv = os.path.join(mimic_root, "machine_measurements.csv")

    record_df = pd.read_csv(record_csv)
    mm_df = pd.read_csv(mm_csv)

    print("record_list shape:", record_df.shape)
    print("machine_measurements shape:", mm_df.shape)

    # local files
    hea_files = sorted(glob.glob(os.path.join(mimic_root, "p*", "**", "*.hea"), recursive=True))
    local_record_paths = [hp[:-4] for hp in hea_files]
    local_rel_paths = [os.path.relpath(p, mimic_root) for p in local_record_paths]
    local_rel_paths = [normalize_record_path(x) for x in local_rel_paths]

    print("Local downloaded .hea files:", len(hea_files))
    print("Local examples:", local_rel_paths[:3])

    # detect path column
    candidate_cols = ["path", "record_path", "file_name", "wfdb_path"]
    path_col = None
    for c in candidate_cols:
        if c in record_df.columns:
            path_col = c
            break

    if path_col is None:
        raise ValueError(f"Could not find path column in record_list.csv. Columns: {record_df.columns.tolist()}")

    print("Using path column:", path_col)
    print("CSV raw examples:", record_df[path_col].head(5).tolist())

    record_df[path_col] = record_df[path_col].astype(str).apply(normalize_record_path)
    print("CSV normalized examples:", record_df[path_col].head(5).tolist())

    local_set = set(local_rel_paths)
    record_df = record_df[record_df[path_col].isin(local_set)].copy()

    print("Matched locally available records:", len(record_df))
    if len(record_df) == 0:
        raise ValueError("Still 0 matched records after normalization.")

    # join with machine measurements
    meta = record_df.merge(mm_df, on=["subject_id", "study_id"], how="left")
    print("Merged shape:", meta.shape)

    # report text from machine-generated summary fields
    report_cols = [c for c in meta.columns if re.fullmatch(r"report_\d+", str(c))]
    report_cols = sorted(report_cols, key=lambda x: int(x.split("_")[1]))

    if len(report_cols) == 0:
        raise ValueError("No report_# columns found in machine_measurements.csv")

    def build_machine_text(row):
        parts = []
        for c in report_cols:
            val = row.get(c, None)
            if pd.notna(val):
                sval = str(val).strip()
                if sval and sval.lower() != "nan":
                    parts.append(sval)
        return " ".join(parts).strip()

    meta["report"] = meta.apply(build_machine_text, axis=1)
    meta["report"] = meta["report"].fillna("").astype(str).str.strip()
    meta = meta[meta["report"] != ""].copy()

    print("Rows with usable machine text:", len(meta))
    print("Sample reports:", meta["report"].head(3).tolist())

    def map_label(text):
        t = str(text).lower()

        if any(x in t for x in ["normal sinus rhythm", "normal ecg", "sinus rhythm", "within normal limits"]):
            return "NORM"
        if any(x in t for x in ["myocardial infarction", "infarct", "stemi", "nstemi", "anterior infarct", "inferior infarct"]):
            return "MI"
        if any(x in t for x in ["atrial fibrillation", "afib", "a-fib", "atrial fib"]):
            return "AFIB"
        if any(x in t for x in ["sinus tachycardia", "tachycardia", "supraventricular tachycardia", "svt"]):
            return "TACHY"
        if any(x in t for x in ["sinus bradycardia", "bradycardia"]):
            return "BRADY"
        return "OTHER"

    meta["chosen_label"] = meta["report"].apply(map_label)

    print("\nRaw label counts:")
    print(meta["chosen_label"].value_counts())

    vc = meta["chosen_label"].value_counts()
    keep = vc[vc >= 5].index.tolist()
    meta = meta[meta["chosen_label"].isin(keep)].copy()

    meta["filename_lr"] = meta[path_col].apply(lambda x: os.path.join(mimic_root, x))

    # balanced subset
    if len(meta) > max_samples:
        n_classes = meta["chosen_label"].nunique()
        per_class = max(1, max_samples // n_classes)
        chunks = []
        for lab in sorted(meta["chosen_label"].unique()):
            part = meta[meta["chosen_label"] == lab]
            chunks.append(part.sample(min(len(part), per_class), random_state=seed))
        meta = pd.concat(chunks).sample(frac=1, random_state=seed).reset_index(drop=True)

        if len(meta) > max_samples:
            meta = meta.sample(max_samples, random_state=seed).reset_index(drop=True)

    # split by study/sample for now
    idx = np.arange(len(meta))
    np.random.shuffle(idx)
    n = len(idx)
    train_idx = set(idx[:int(0.7 * n)])
    val_idx = set(idx[int(0.7 * n):int(0.85 * n)])
    test_idx = set(idx[int(0.85 * n):])

    meta["split"] = [
        "train" if i in train_idx else ("val" if i in val_idx else "test")
        for i in range(len(meta))
    ]

    label_names = sorted(meta["chosen_label"].unique())
    label_map = {lab: i for i, lab in enumerate(label_names)}

    out_cols = ["subject_id", "study_id", "filename_lr", "report", "chosen_label", "split"]
    meta = meta[out_cols].reset_index(drop=True)

    print("\nFinal size:", len(meta))
    print("\nFinal label counts:")
    print(meta["chosen_label"].value_counts())
    print("\nSplit counts:")
    print(meta["split"].value_counts())
    print("\nLabel map:", label_map)

    return meta, label_map

mimic_meta, mimic_label_map = build_mimic_metadata_from_csv(
    MIMIC_ROOT,
    max_samples=200,
    seed=42
)

print(mimic_meta.head())

/tmp/ipykernel_188096/6698481.py:31: DtypeWarning: Columns (16,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  mm_df = pd.read_csv(mm_csv)


record_list shape: (800035, 5)
machine_measurements shape: (800035, 33)
Local downloaded .hea files: 796
Local examples: ['p1000/p10000032/s40689238/40689238', 'p1000/p10000032/s44458630/44458630', 'p1000/p10000032/s49036311/49036311']
Using path column: path
CSV raw examples: ['files/p1000/p10000032/s40689238/40689238', 'files/p1000/p10000032/s44458630/44458630', 'files/p1000/p10000032/s49036311/49036311', 'files/p1000/p10000117/s45090959/45090959', 'files/p1000/p10000117/s48446569/48446569']
CSV normalized examples: ['p1000/p10000032/s40689238/40689238', 'p1000/p10000032/s44458630/44458630', 'p1000/p10000032/s49036311/49036311', 'p1000/p10000117/s45090959/45090959', 'p1000/p10000117/s48446569/48446569']
Matched locally available records: 795
Merged shape: (795, 36)
Rows with usable machine text: 795
Sample reports: ['Sinus rhythm Possible right atrial abnormality Borderline ECG', 'Sinus rhythm Possible right atrial abnormality Borderline ECG', 'Sinus tachycardia Normal ECG except for

In [12]:
mimic_meta, mimic_label_map = build_mimic_metadata_from_csv(
    MIMIC_ROOT,
    max_samples=500,
    seed=42
)

print("Total samples:", len(mimic_meta))
print(mimic_meta["chosen_label"].value_counts())
print(mimic_meta["split"].value_counts())
print(mimic_label_map)

/tmp/ipykernel_188096/6698481.py:31: DtypeWarning: Columns (16,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  mm_df = pd.read_csv(mm_csv)


record_list shape: (800035, 5)
machine_measurements shape: (800035, 33)
Local downloaded .hea files: 4715
Local examples: ['p1000/p10000032/s40689238/40689238', 'p1000/p10000032/s44458630/44458630', 'p1000/p10000032/s49036311/49036311']
Using path column: path
CSV raw examples: ['files/p1000/p10000032/s40689238/40689238', 'files/p1000/p10000032/s44458630/44458630', 'files/p1000/p10000032/s49036311/49036311', 'files/p1000/p10000117/s45090959/45090959', 'files/p1000/p10000117/s48446569/48446569']
CSV normalized examples: ['p1000/p10000032/s40689238/40689238', 'p1000/p10000032/s44458630/44458630', 'p1000/p10000032/s49036311/49036311', 'p1000/p10000117/s45090959/45090959', 'p1000/p10000117/s48446569/48446569']
Matched locally available records: 4714
Merged shape: (4714, 36)
Rows with usable machine text: 4714
Sample reports: ['Sinus rhythm Possible right atrial abnormality Borderline ECG', 'Sinus rhythm Possible right atrial abnormality Borderline ECG', 'Sinus tachycardia Normal ECG except

In [ ]:
#python /home/asatsan2/Projects/ECG/pipeline.py --dataset mimic --data_dir /home/asatsan2/Projects/ECG/mimic_ecg --epochs 5 --batch_size 16 --mimic_max_samples 1000

In [ ]:
"""ArithmeticError

python /home/asatsan2/Projects/ECG/pipeline.py --dataset mimic --data_dir /home/asatsan2/Projects/ECG/mimic_ecg --epochs 5 --batch_size 16 --mimic_max_samples 1000


"""

In [13]:
@torch.no_grad()
def export_mimic_retrieval_examples(
    model,
    retrieval_db,
    dataset,
    tokenizer,
    device,
    config,
    out_csv="mimic_retrieval_examples.csv",
    num_examples=3,
):
    model.eval()

    loader = DataLoader(
        dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        collate_fn=custom_collate_fn,
    )

    rows = []
    inv_label_map = {v: k for k, v in dataset.label_map.items()}

    for i, (ecg, text_enc, labels) in enumerate(loader):
        ecg = ecg.to(device)
        text_enc = {k: v.to(device) for k, v in text_enc.items()}
        labels = labels.to(device)

        q_ecg = model.encode_ecg(ecg).mean(dim=1)
        q_txt = model.encode_text_vector(text_enc)

        retrieved_reports = retrieval_db.retrieve(q_ecg, q_txt, k=3, alpha=0.6)[0]
        retrieved_text_encs = prepare_retrieved_text_batch(
            [retrieved_reports], tokenizer, device, config.TEXT_MAX_LEN
        )

        logits = model(ecg, text_enc, retrieved_text_encs)
        probs = torch.softmax(logits, dim=1)

        pred_idx = probs.argmax(dim=1).item()
        true_idx = labels.item()

        meta_row = dataset.meta.iloc[i]

        rows.append({
            "sample_idx": i,
            "filename_lr": meta_row["filename_lr"],
            "true_label": inv_label_map[true_idx],
            "pred_label": inv_label_map[pred_idx],
            "orig_report": meta_row["report"],
            "retrieved_1": retrieved_reports[0] if len(retrieved_reports) > 0 else "",
            "retrieved_2": retrieved_reports[1] if len(retrieved_reports) > 1 else "",
            "retrieved_3": retrieved_reports[2] if len(retrieved_reports) > 2 else "",
        })

        if len(rows) >= num_examples:
            break

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"Saved retrieval examples to {out_csv}")
    print(df)
    return df

NameError: name 'torch' is not defined

In [11]:
mimic_meta, mimic_label_map = build_mimic_metadata_from_csv(
    MIMIC_ROOT,
    max_samples=500,
    seed=42
)

print("Total samples:", len(mimic_meta))
print(mimic_meta["chosen_label"].value_counts())
print(mimic_meta["split"].value_counts())
print(mimic_label_map)

/tmp/ipykernel_188096/6698481.py:31: DtypeWarning: Columns (16,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  mm_df = pd.read_csv(mm_csv)


record_list shape: (800035, 5)
machine_measurements shape: (800035, 33)
Local downloaded .hea files: 4646
Local examples: ['p1000/p10000032/s40689238/40689238', 'p1000/p10000032/s44458630/44458630', 'p1000/p10000032/s49036311/49036311']
Using path column: path
CSV raw examples: ['files/p1000/p10000032/s40689238/40689238', 'files/p1000/p10000032/s44458630/44458630', 'files/p1000/p10000032/s49036311/49036311', 'files/p1000/p10000117/s45090959/45090959', 'files/p1000/p10000117/s48446569/48446569']
CSV normalized examples: ['p1000/p10000032/s40689238/40689238', 'p1000/p10000032/s44458630/44458630', 'p1000/p10000032/s49036311/49036311', 'p1000/p10000117/s45090959/45090959', 'p1000/p10000117/s48446569/48446569']
Matched locally available records: 4174
Merged shape: (4174, 36)
Rows with usable machine text: 4174
Sample reports: ['Sinus rhythm Possible right atrial abnormality Borderline ECG', 'Sinus rhythm Possible right atrial abnormality Borderline ECG', 'Sinus tachycardia Normal ECG except

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -----------------------------
# Data
# -----------------------------
ptbxl_data = [
    ["ECG Only", 0.5221, 0.1843, 0.8541],
    ["DistilBERT", 0.8653, 0.2122, 0.8586],
    ["BioBERT", 0.9233, 0.2120, 0.8582],
    ["BiomedBERT", 0.8732, 0.2121, 0.8593],
    ["DistilBERT + Cross-Attn", 0.9871, 0.5860, 0.9574],
    ["BioBERT + Cross-Attn", 0.9911, 0.6174, 0.9635],
    ["BiomedBERT + Cross-Attn", 0.9877, 0.6186, 0.9587],
    ["DistilBERT + RAG (K=1)", 0.9870, 0.6346, 0.9472],
    ["DistilBERT + RAG (K=3)", 0.9875, 0.6408, 0.9562],
    ["DistilBERT + RAG (K=5)", 0.9878, 0.5787, 0.9493],
    ["BioBERT + RAG (K=1)", 0.9896, 0.7077, 0.9524],
    ["BioBERT + RAG (K=3)", 0.9930, 0.6388, 0.9658],
    ["BioBERT + RAG (K=5)", 0.9882, 0.6495, 0.9572],
    ["BiomedBERT + RAG (K=1)", 0.9891, 0.6409, 0.9591],
    ["BiomedBERT + RAG (K=3)", 0.9875, 0.6197, 0.9581],
    ["BiomedBERT + RAG (K=5)", 0.9881, 0.6502, 0.9574],
]

mimic_data = [
    ["ECG Only", 0.6826, 0.4151, 0.5439],
    ["DistilBERT", 0.8717, 0.5647, 0.5877],
    ["BioBERT", 0.8719, 0.6635, 0.6491],
    ["BiomedBERT", 0.8599, 0.5481, 0.5526],
    ["DistilBERT + Cross-Attn", 0.9791, 0.8798, 0.8772],
    ["BioBERT + Cross-Attn", 0.9857, 0.9081, 0.9191],
    ["BiomedBERT + Cross-Attn", 0.9441, 0.7790, 0.7982],
    ["DistilBERT + RAG (K=1)", 0.9783, 0.8712, 0.8684],
    ["DistilBERT + RAG (K=3)", 0.9783, 0.8522, 0.8509],
    ["DistilBERT + RAG (K=5)", 0.9771, 0.8443, 0.8421],
    ["BioBERT + RAG (K=1)", 0.9903, 0.9050, 0.9123],
    ["BioBERT + RAG (K=3)", 0.9906, 0.9179, 0.9211],
    ["BioBERT + RAG (K=5)", 0.9916, 0.9098, 0.9123],
    ["BiomedBERT + RAG (K=1)", 0.9278, 0.7236, 0.7456],
    ["BiomedBERT + RAG (K=3)", 0.9256, 0.7442, 0.7632],
    ["BiomedBERT + RAG (K=5)", 0.9186, 0.7371, 0.7544],
]

# -----------------------------
# Helper functions
# -----------------------------
def make_df(data):
    return pd.DataFrame(data, columns=["Model", "AUROC", "F1", "Accuracy"])

def add_heatmap_values(ax, arr):
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            ax.text(j, i, f"{arr[i, j]:.3f}", ha="center", va="center", fontsize=8)

def plot_heatmap(df, title, out_pdf):
    models = df["Model"].tolist()
    metrics = ["AUROC", "F1", "Accuracy"]
    values = df[metrics].values

    fig, ax = plt.subplots(figsize=(8.5, 7.5))
    im = ax.imshow(values, aspect="auto", vmin=0.0, vmax=1.0)

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xticks(np.arange(len(metrics)))
    ax.set_xticklabels(metrics, fontsize=11)
    ax.set_yticks(np.arange(len(models)))
    ax.set_yticklabels(models, fontsize=9)

    add_heatmap_values(ax, values)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Score", fontsize=11)

    plt.tight_layout()
    plt.savefig(out_pdf, format="pdf", bbox_inches="tight", dpi=300)
    plt.close()

def plot_bar(df, title, out_pdf):
    x = np.arange(len(df))
    width = 0.25

    fig, ax = plt.subplots(figsize=(16, 7))
    ax.bar(x - width, df["AUROC"], width, label="AUROC")
    ax.bar(x, df["F1"], width, label="F1")
    ax.bar(x + width, df["Accuracy"], width, label="Accuracy")

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel("Score", fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(df["Model"], rotation=45, ha="right", fontsize=9)
    ax.legend(fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.savefig(out_pdf, format="pdf", bbox_inches="tight", dpi=300)
    plt.close()

# -----------------------------
# Build dataframes
# -----------------------------
ptbxl_df = make_df(ptbxl_data)
mimic_df = make_df(mimic_data)

# -----------------------------
# Generate heatmaps
# -----------------------------
plot_heatmap(
    ptbxl_df,
    "PTB-XL Performance Heatmap",
    "ptbxl_heatmap.pdf"
)

plot_heatmap(
    mimic_df,
    "MIMIC-IV-ECG Performance Heatmap",
    "mimic_heatmap.pdf"
)

# -----------------------------
# Generate grouped bar charts
# -----------------------------
plot_bar(
    ptbxl_df,
    "PTB-XL Performance Comparison",
    "ptbxl_barplot.pdf"
)

plot_bar(
    mimic_df,
    "MIMIC-IV-ECG Performance Comparison",
    "mimic_barplot.pdf"
)

print("Saved:")
print(" - ptbxl_heatmap.pdf")
print(" - mimic_heatmap.pdf")
print(" - ptbxl_barplot.pdf")
print(" - mimic_barplot.pdf")

Saved:
 - ptbxl_heatmap.pdf
 - mimic_heatmap.pdf
 - ptbxl_barplot.pdf
 - mimic_barplot.pdf


In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# -----------------------------
# Representative PTB-XL models
# -----------------------------
ptbxl_data = [
    ["ECG Only", 0.5221, 0.1843, 0.8541],
    ["Text Only DistilBERT", 0.8653, 0.2122, 0.8586],
    ["Text Only BioBERT", 0.9233, 0.2120, 0.8582],
    ["Text Only BiomedBERT", 0.8732, 0.2121, 0.8593],
    ["DistilBERT + Cross-Attn", 0.9871, 0.5860, 0.9574],
    ["BioBERT + Cross-Attn", 0.9911, 0.6174, 0.9635],
    ["BiomedBERT + Cross-Attn", 0.9877, 0.6186, 0.9587],
    ["DistilBERT + RAG (K=3)", 0.9875, 0.6408, 0.9562],
    ["BioBERT + RAG (K=3)", 0.9930, 0.6388, 0.9658],
    ["BiomedBERT + RAG (K=3)", 0.9875, 0.6197, 0.9581],
]

# -----------------------------
# Representative MIMIC models
# -----------------------------
mimic_data = [
    ["ECG Only", 0.6826, 0.4151, 0.5439],
    ["Text Only DistilBERT", 0.8717, 0.5647, 0.5877],
    ["Text Only BioBERT", 0.8719, 0.6635, 0.6491],
    ["Text Only BiomedBERT", 0.8599, 0.5481, 0.5526],
    ["DistilBERT + Cross-Attn", 0.9791, 0.8798, 0.8772],
    ["BioBERT + Cross-Attn", 0.9857, 0.9081, 0.9191],
    ["BiomedBERT + Cross-Attn", 0.9441, 0.7790, 0.7982],
    ["DistilBERT + RAG (K=3)", 0.9783, 0.8522, 0.8509],
    ["BioBERT + RAG (K=3)", 0.9906, 0.9179, 0.9211],
    ["BiomedBERT + RAG (K=3)", 0.9256, 0.7442, 0.7632],
]


def make_df(data):
    return pd.DataFrame(data, columns=["Model", "AUROC", "F1", "Accuracy"])


def add_heatmap_values(ax, arr):
    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            ax.text(j, i, f"{arr[i, j]:.3f}", ha="center", va="center", fontsize=8)


def plot_heatmap(df, title, out_pdf):
    models = df["Model"].tolist()
    metrics = ["AUROC", "F1", "Accuracy"]
    values = df[metrics].values

    fig, ax = plt.subplots(figsize=(8.8, 5.8))
    im = ax.imshow(values, aspect="auto", vmin=0.0, vmax=1.0)

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xticks(np.arange(len(metrics)))
    ax.set_xticklabels(metrics, fontsize=11)
    ax.set_yticks(np.arange(len(models)))
    ax.set_yticklabels(models, fontsize=10)

    add_heatmap_values(ax, values)

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Score", fontsize=11)

    plt.tight_layout()
    plt.savefig(out_pdf, format="pdf", bbox_inches="tight", dpi=300)
    plt.close()


def plot_bar(df, title, out_pdf):
    x = np.arange(len(df))
    width = 0.24

    fig, ax = plt.subplots(figsize=(12, 5.8))
    ax.bar(x - width, df["AUROC"], width, label="AUROC")
    ax.bar(x, df["F1"], width, label="F1")
    ax.bar(x + width, df["Accuracy"], width, label="Accuracy")

    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_ylabel("Score", fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(df["Model"], rotation=35, ha="right", fontsize=9)
    ax.legend(fontsize=10)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.savefig(out_pdf, format="pdf", bbox_inches="tight", dpi=300)
    plt.close()


ptbxl_df = make_df(ptbxl_data)
mimic_df = make_df(mimic_data)

plot_heatmap(ptbxl_df, "PTB-XL: Representative Model Comparison", "ptbxl_rep_heatmap.pdf")
plot_heatmap(mimic_df, "MIMIC-IV-ECG: Representative Model Comparison", "mimic_rep_heatmap.pdf")

plot_bar(ptbxl_df, "PTB-XL: Representative Model Comparison", "ptbxl_rep_barplot.pdf")
plot_bar(mimic_df, "MIMIC-IV-ECG: Representative Model Comparison", "mimic_rep_barplot.pdf")

print("Saved:")
print(" - ptbxl_rep_heatmap.pdf")
print(" - mimic_rep_heatmap.pdf")
print(" - ptbxl_rep_barplot.pdf")
print(" - mimic_rep_barplot.pdf")

Saved:
 - ptbxl_rep_heatmap.pdf
 - mimic_rep_heatmap.pdf
 - ptbxl_rep_barplot.pdf
 - mimic_rep_barplot.pdf


In [7]:
print("Final size:", len(mimic_meta))
print(mimic_meta["chosen_label"].value_counts())
print(mimic_meta["split"].value_counts())
print(mimic_label_map)

Final size: 139
chosen_label
NORM     50
TACHY    49
BRADY    27
OTHER    13
Name: count, dtype: int64
split
train    97
test     21
val      21
Name: count, dtype: int64
{'BRADY': 0, 'NORM': 1, 'OTHER': 2, 'TACHY': 3}
